# Dioptra-DINO: Multi-Domain Large-Scale Metric Depth Training

Training **Dioptra-DINO** (~27.5M params) on dual NVIDIA T4 GPUs across an aggregated multi-domain corpus:
- **TartanAir & TartanGround AMR**: 8 public stereo suites (Warehouse, Hospital, Office, OldIndustrialCity, Restaurant, School)
- **Apple Hypersim**: 191 indoor environments (`volsiai/hypersim-pack`)
- **NYU-Depth-v2**: Official RGB-D indoor benchmark (`soumikrakshit/nyu-depth-v2`)
- **KITTI**: Eigen metric depth benchmark (`alextitto/kitti-rgb-depth-20k-subset`)
- **DASVO TartanAir**: Benchmark validation split (`pandrii000/dasvo-tartanair-rgb-d-validation-split`)
- **Training Mode**: From-scratch joint multi-domain training (Epoch 0 to 40) using pre-trained DINOv2-Small ViT backbone.

In [ ]:
# [1] Deploy latest Multi-Domain Dioptra-DINO codebase
import os, sys, base64

CODE_B64 = """..."""
with open('/kaggle/working/dioptra_dino.py', 'wb') as f:
    f.write(base64.b64decode(CODE_B64))

if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

sz = os.path.getsize('/kaggle/working/dioptra_dino.py')
print(f'>>> Deployed multi-domain dioptra_dino.py to /kaggle/working/ ({sz:,} bytes) <<<')


In [ ]:
# [2] Multi-Domain Dataset Discovery & Audit
import os, sys, glob
from dioptra_dino import resolve_all_dataset_roots, MultiDomainDINODataset

print('Scanning all mounted input datasets in /kaggle/input/...')
roots = resolve_all_dataset_roots('auto')
print(f'Discovered {len(roots)} multi-domain dataset roots:')
for r, dom in roots:
    print(f'  [{dom.upper():8s}] {r}')

print('\nIndexing training dataset across all domains...')
train_dataset = MultiDomainDINODataset(root_dirs='auto', split='train', image_size=224, apply_pinhole_aug=True, crop_min=0.35)
print(f'Total training samples: {len(train_dataset):,}')

dom_counts = {}
for s in train_dataset.samples:
    dom_counts[s[2]] = dom_counts.get(s[2], 0) + 1
print('\nTraining domain breakdown:')
for dom, cnt in sorted(dom_counts.items()):
    print(f'  {dom.upper():8s}: {cnt:,} samples')


In [ ]:
# [3] Launch Training from Scratch (Epoch 0 to 40)
# --train auto : Auto-scans all 12 mounted datasets across TartanAir, Hypersim, NYUv2, and KITTI
# --resume none : Trains from scratch (Epoch 0) initializing from official DINOv2-Small ViT weights
# --weight-normal 0.25 : 3D Virtual Normal Loss enforcing surface planarity & boundary sharpness
# --crop-min 0.35 : Wide optical zoom crop for camera-intrinsic equivariance
# --batch-size 8 : Batch size per GPU (DataParallel multi-GPU acceleration across 2x T4s)
import os

cmd = (
    'python /kaggle/working/dioptra_dino.py '
    '--train auto '
    '--epochs 40 '
    '--batch-size 8 '
    '--weight-normal 0.25 '
    '--crop-min 0.35 '
    '--lr-backbone 2e-5 '
    '--lr-head 2e-4 '
    '--resume none '
    '--output-dir /kaggle/working/outputs_dino'
)
print('Executing training command:')
print(cmd)
print('=' * 70)
exit_code = os.system(cmd)
assert exit_code == 0, f'Training failed with exit code {exit_code}'


In [ ]:
# [4] Verify Checkpoints
import os, glob

output_dir = '/kaggle/working/outputs_dino'
if os.path.exists(output_dir):
    ckpts = sorted(glob.glob(os.path.join(output_dir, '*.pt')))
    print(f'Checkpoints in {output_dir} ({len(ckpts)} found):')
    for cp in ckpts:
        sz_mb = os.path.getsize(cp) / (1024 * 1024)
        print(f'  {os.path.basename(cp)} ({sz_mb:.2f} MB)')
else:
    print('outputs_dino directory not found.')


In [ ]:
# [5] Package Checkpoints & Training Artifacts
import os
from IPython.display import display, FileLink

os.system('cd /kaggle/working && zip -q -r dioptra_dino_multidomain_checkpoints.zip outputs_dino/')
archive_path = '/kaggle/working/dioptra_dino_multidomain_checkpoints.zip'
if os.path.exists(archive_path):
    sz_mb = os.path.getsize(archive_path) / (1024 * 1024)
    print(f'>>> Successfully packaged all checkpoints! Archive size: {sz_mb:.2f} MB <<<')
    display(FileLink('dioptra_dino_multidomain_checkpoints.zip'))
